# 04. lgbm 강화: 튜닝, 교차검증, 예측 근거, 확률 신뢰도

03에서 고른 lgbm을 네 가지로 검증하고 강화한다.

| 단계 | 질문 |
|---|---|
| ① 하이퍼파라미터 튜닝 | 기본 설정보다 더 좋은 설정이 있나? |
| ② 시계열 교차검증 | 운 좋은 4일이 아니라 여러 기간에서 꾸준히 맞나? TMAP 도착 시간(5분 단위 아무 값)에서도 맞나? |
| ③ 예측 근거 (SHAP) | 모델이 무엇을 보고 예측하나? |
| ④ 확률 신뢰도 | "자리 있을 확률 80%"가 실제로 80% 맞나? |

공정성 규칙: 튜닝은 검증 기간만 보고, 평가 기간은 튜닝이 끝난 뒤 한 번만 본다.
전체 45~60분 걸린다 (① 30분, ② 10~20분).

## 1. 설정

In [ ]:
import os, sys, json
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
sys.path.insert(0, os.path.abspath(".."))
import logging
import pandas as pd, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from jariml import load_config, tuning
from jariml.benchmark import run_benchmark
from jariml.engines import ENGINES

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
plt.rcParams["font.family"] = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]
                                    if f in {x.name for x in fm.fontManager.ttflist}), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False
cfg = load_config()
tables, sp = tuning.load(cfg)
sp.describe()

## 2. ① 하이퍼파라미터 튜닝 (Optuna)

검증 기간 1시간 뒤 MAE(대)가 가장 작은 설정을 찾는다. 1번 시도는 지금 설정이라 튜닝 결과가 지금보다 나빠질 수 없다.

학습 손실함수(l2, l1, huber)도 후보에 넣어 함께 고른다. 설정 고르는 기준은 검증 기간 1시간 뒤 빈자리 MAE(대)다.

`optuna`가 없으면 먼저 `%pip install optuna` 실행 후 커널 재시작.

In [ ]:
N_TRIALS, TIMEOUT_MIN = 60, 30          # 시도 횟수, 최대 시간 (먼저 닿는 쪽에서 멈춤)
study = tuning.tune_lgbm(cfg, tables, sp, N_TRIALS, TIMEOUT_MIN)
best = study.best_params
cur = study.trials[0].params
DESC = {"objective": "손실함수 (l2=MSE, l1=MAE, huber)", "huber_delta": "huber 경계 (점유율 변화량)", "num_leaves": "나무 하나의 복잡도",
        "learning_rate": "한 번에 배우는 양", "min_child_samples": "잎 하나 최소 표본", "subsample": "나무마다 쓰는 행 비율",
        "colsample_bytree": "나무마다 쓰는 변수 비율", "reg_lambda": "가중치 규제"}
pd.DataFrame({"지금 설정": pd.Series(cur), "튜닝 결과": pd.Series(best), "설명": pd.Series(DESC)}).loc[[k for k in DESC if k in cur or k in best]]

In [ ]:
v = pd.Series([t.value for t in study.trials])
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(v.index + 1, v, s=15, alpha=0.5, label="각 시도")
ax.plot(v.index + 1, v.cummin(), color="C3", label="지금까지 최고")
ax.axhline(v[0], ls="--", color="gray", lw=1); ax.text(len(v), v[0], "지금 설정", ha="right", va="bottom", color="gray")
ax.set(xlabel="시도", ylabel="검증 기간 1시간 MAE(대)", title=f"튜닝 과정: {v[0]:.3f}대 → {v.min():.3f}대 ({(v.min() / v[0] - 1) * 100:+.1f}%)")
ax.legend(); ax.grid(alpha=0.3); plt.show()

## 3. 평가 기간에서 한 번 확인

튜닝이 끝났으니 처음으로 평가 기간을 본다. 세 번째는 튜닝 설정에 더해 예측 거리 5분~2시간 24칸을 모두 학습한 버전이다 (지금은 8칸만 학습).
TMAP 도착 시간이 35분, 50분처럼 학습에 없던 값으로 나와도 바로 맞히게 하려는 것이다.

In [ ]:
tuned = tuning.with_params(cfg, best)
tuned_all = tuning.with_params(cfg, best)
tuned_all["target"]["horizons"] = list(range(1, cfg["target"]["max_horizon_steps"] + 1))
VARIANTS = {"lgbm 기본": cfg, "lgbm 튜닝": tuned, "lgbm 튜닝 + 24칸 학습": tuned_all}

res = []
for name, c in VARIANTS.items():
    r = run_benchmark(c, ["lgbm"], repeat=10, data=(tables, sp))["result"]
    res.append(r.rename(index={"lgbm": name}))
test = pd.concat([res[0]] + [r.loc[[i for i in r.index if not i.startswith("(")]] for r in res[1:]])
test[["MAE 30분(대)", "MAE 1시간(대)", "MAE 2시간(대)", "RMSE 1시간(대)", "MSLE 1시간", "Brier", "예측 응답(ms)", "학습(초)"]].round(3)

## 4. ② 시계열 교차검증

평가 기간을 2일씩 과거로 옮기며 4번 평가한다. 각 회차는 그 시점까지의 데이터만 쓴다.
"5분~2시간 평균"은 TMAP이 어떤 도착 시간을 주든 평균적으로 몇 대 틀리는지를 뜻한다.

In [ ]:
wf, wf_sum = tuning.walk_forward(cfg, tables, {n: ("lgbm", c) for n, c in VARIANTS.items()}, n_folds=4, fold_days=2)
wf.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["1시간 MAE(대)", "5분~2시간 평균 MAE(대)"]):
    m = wf_sum.loc[metric]
    ax.barh(m.index, m["평균"], xerr=m["표준편차"], capsize=4, color=["gray"] + [f"C{i}" for i in range(len(m) - 1)])
    for i, (a, s) in enumerate(zip(m["평균"], m["표준편차"])):
        ax.text(a + s + 0.05, i, f"{a:.2f} ± {s:.2f}", va="center")
    ax.set(title=metric, xlabel="대 (막대 = 평균, 선 = 표준편차)"); ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()
wf_sum.round(3)

## 5. ③ 예측 근거 (SHAP)

교차검증에서 가장 좋은 버전으로, 1시간 뒤 예측에서 각 변수가 빈자리 수를 평균 몇 대 움직였는지 본다.

In [ ]:
wf_best = wf_sum.loc["1시간 MAE(대)"]["평균"].drop("(기준선) 지금+평소변화").idxmin()
final_cfg = VARIANTS[wf_best]
eng = ENGINES["lgbm"](final_cfg)
eng.fit(tables.occ, tables.cap, sp)
imp, S, X = tuning.shap_table(eng, tables, sp, final_cfg)
print("최종 버전:", wf_best)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
imp.sort_values().plot.barh(ax=axes[0], color="C0")
axes[0].set(title="변수별 평균 영향 (1시간 뒤 빈자리, 대)", xlabel="대")
top = imp.index[0]
axes[1].scatter(X[top], S[top], s=3, alpha=0.3)
axes[1].axhline(0, color="gray", lw=1)
axes[1].set(title=f"가장 중요한 변수: {top}", xlabel=top, ylabel="빈자리에 준 영향(대, + = 늘어남)")
plt.tight_layout(); plt.show()
imp.round(3).to_frame()

## 6. ④ 확률 신뢰도

예측 확률 구간별로 실제 자리가 있었던 비율을 본다. 점이 대각선에 가까울수록 확률을 믿을 수 있다.

In [ ]:
prob, cal = tuning.calibration(eng, tables, sp, final_cfg)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], ls="--", color="gray", label="완벽")
ax.plot(cal["예측확률평균"], cal["실제비율"], marker="o", label="우리 모델")
for _, r in cal.iterrows():
    ax.annotate(f"{r['건수']:,}건", (r["예측확률평균"], r["실제비율"]), xytext=(5, -12), textcoords="offset points", fontsize=8)
ax.set(xlabel="예측한 확률", ylabel="실제 자리 있던 비율", title=f"확률 신뢰도 (Brier {prob['Brier']:.4f}, 평균만 쓰면 {prob['Brier(항상 평균 확률)']:.4f})")
ax.legend(); ax.grid(alpha=0.3); plt.show()
cal.round(3)

## 7. 최종 설정 저장

결과를 `ml/reports/lgbm_final.json`에 저장한다. 이 파일 내용을 공유하면 config.yaml에 반영한다.

In [ ]:
os.makedirs("../reports", exist_ok=True)
out = {"version": wf_best, "params": best, "horizons": final_cfg["target"]["horizons"],
       "test": test.loc[wf_best, ["MAE 30분(대)", "MAE 1시간(대)", "MAE 2시간(대)", "RMSE 1시간(대)", "MSLE 1시간", "Brier", "예측 응답(ms)"]].round(4).to_dict(),
       "walk_forward_1h": wf_sum.loc["1시간 MAE(대)"].round(4).to_dict("index")}
json.dump(out, open("../reports/lgbm_final.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
print(json.dumps(out, ensure_ascii=False, indent=1, default=float))